[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nahuelcostac/generative-networks/blob/main/reduction/lora.ipynb)

This notebook is part of the [Generative Networks course](https://www.nahuelcosta.com/teaching/generative-networks).

# Adapting a Language Model with LoRA and QLoRA

In this tutorial we adapt the 360M-parameter SmolLM2 base model for summarization while training only low-rank adapters. The default path is **QLoRA**: the frozen base is loaded in 4-bit NF4 and the LoRA parameters remain trainable at higher precision.

We will:

- inspect the low-rank parameterization;
- stream and format a conversational summarization dataset;
- target the query and value projections with PEFT;
- measure trainable parameters and memory;
- train with TRL's `SFTTrainer`;
- save, reload, and merge the adapter;
- verify why a tiny adapter does not imply a tiny merged model.

## 0. Dependencies

Package metadata is used instead of `find_spec` because this repository contains a local directory named `transformers/`. The cell pins the validated course environment without replacing Colab's CUDA-enabled PyTorch.

In [ ]:
from importlib import metadata
import subprocess
import sys

PINNED_PACKAGES = {
    "transformers": "5.14.1",
    "datasets": "5.0.0",
    "accelerate": "1.14.0",
    "peft": "0.19.1",
    "trl": "1.9.0",
    "bitsandbytes": "0.49.2",
}

to_install = []
for distribution, expected_version in PINNED_PACKAGES.items():
    try:
        installed_version = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        installed_version = None
    if installed_version != expected_version:
        to_install.append(f"{distribution}=={expected_version}")

if to_install:
    print("Installing the validated course environment:", ", ".join(to_install))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *to_install])
else:
    print("Validated dependencies are already installed.")

for distribution, expected_version in PINNED_PACKAGES.items():
    actual_version = metadata.version(distribution)
    print(f"{distribution}: {actual_version}")
    assert actual_version == expected_version

## 1. Imports and Reproducibility

In [ ]:
import gc
import math
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from trl import SFTConfig, SFTTrainer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Configuration

`USE_QLORA=True` is the default and requires a CUDA GPU. If CUDA is unavailable, the notebook falls back to ordinary full-precision LoRA so the mechanics remain inspectable. Set `FAST_DEV_RUN=1` to reduce training to two steps.

In [ ]:
fast_dev_run = os.environ.get("FAST_DEV_RUN", "0") == "1"
requested_qlora = os.environ.get("USE_QLORA", "1") == "1"
use_qlora = requested_qlora and torch.cuda.is_available()
if requested_qlora and not use_qlora:
    print("CUDA is unavailable; falling back from QLoRA to ordinary LoRA.")

BASE_MODEL_ID = "HuggingFaceTB/SmolLM2-360M"
TOKENIZER_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
DATASET_ID = "HuggingFaceTB/smoltalk"
DATASET_CONFIG = "smol-summarize"

project_root = Path.cwd().parent if Path.cwd().name == "reduction" else Path.cwd()
output_dir = project_root / "reduction" / "outputs" / "lora"
adapter_dir = output_dir / "summarization_adapter"
merged_dir = output_dir / "merged_model"
output_dir.mkdir(parents=True, exist_ok=True)

train_examples = 16 if fast_dev_run else 512
evaluation_examples = 8 if fast_dev_run else 64
max_steps = 2 if fast_dev_run else 100
per_device_batch_size = 1 if fast_dev_run else 4
gradient_accumulation_steps = 1 if fast_dev_run else 4
max_length = 128 if fast_dev_run else 256
learning_rate = 1e-4

LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj"]

print("FAST_DEV_RUN:", fast_dev_run)
print("Training mode:", "QLoRA (NF4 base)" if use_qlora else "LoRA")
print("Artifacts:", output_dir)

## 3. Low-Rank Updates

For a frozen weight matrix `W0` with shape `(d_out, d_in)`, LoRA learns `B @ A` where `A` has shape `(r, d_in)` and `B` has shape `(d_out, r)`. The forward pass uses:

`W0 x + (alpha / r) B A x`

When `r` is much smaller than the input and output widths, the trainable update contains far fewer parameters than a full copy of `W0`. PEFT initializes the update so the wrapped model initially behaves like the base model.

In [ ]:
d_in = d_out = 4_096
full_update_parameters = d_in * d_out
lora_update_parameters = LORA_RANK * (d_in + d_out)
print(f"Full update: {full_update_parameters:,} parameters")
print(f"LoRA rank-{LORA_RANK} update: {lora_update_parameters:,} parameters")
print(f"Fraction trained: {100 * lora_update_parameters / full_update_parameters:.3f}%")
assert lora_update_parameters < 0.01 * full_update_parameters

## 4. Stream and Format Smol-Summarize

The dataset contains structured conversations. We take a deterministic bounded sample without downloading the complete corpus, then split each conversation at its final assistant turn. TRL recognizes the resulting conversational prompt/completion format and computes loss only on the completion.

In [ ]:
def take_streaming_rows(split, count):
    stream = load_dataset(
        DATASET_ID, DATASET_CONFIG, split=split, streaming=True
    ).shuffle(seed=SEED, buffer_size=1_000)
    return list(stream.take(count))

def to_prompt_completion(row):
    messages = row["messages"]
    assistant_indices = [
        index for index, message in enumerate(messages)
        if message["role"] == "assistant"
    ]
    if not assistant_indices:
        return None
    final_assistant = assistant_indices[-1]
    prompt = messages[:final_assistant]
    completion = messages[final_assistant:]
    if not prompt or not completion:
        return None
    return {"prompt": prompt, "completion": completion}

def build_dataset(split, count):
    formatted = []
    requested = count
    while len(formatted) < count:
        rows = take_streaming_rows(split, requested)
        formatted = [item for row in rows if (item := to_prompt_completion(row))]
        requested *= 2
    return Dataset.from_list(formatted[:count])

train_dataset = build_dataset("train", train_examples)
evaluation_dataset = build_dataset("test", evaluation_examples)

print(train_dataset, evaluation_dataset)
print(train_dataset[0])
assert len(train_dataset) == train_examples
assert len(evaluation_dataset) == evaluation_examples

## 5. Tokenizer and Baseline Prompt

The base and instruct checkpoints share model vocabulary, while the instruct tokenizer supplies the chat template used during post-training. We explicitly verify the vocabulary size before pairing them.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
assert len(tokenizer) == len(base_tokenizer)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

fixed_messages = [{
    "role": "user",
    "content": (
        "Summarize in one sentence: A neighborhood library extended its opening "
        "hours after a volunteer program added evening staff. Attendance rose by "
        "thirty percent during the first month."
    ),
}]
fixed_prompt = tokenizer.apply_chat_template(
    fixed_messages, tokenize=False, add_generation_prompt=True
)
print(fixed_prompt)

## 6. Load the Base Model

The QLoRA path stores frozen linear weights in 4-bit NF4, uses nested quantization for their metadata, and computes matrix operations in float16. Only the adapter parameters will be optimized.

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

if use_qlora:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=quantization_config,
        device_map={"": 0},
        dtype=torch.float16,
    )
    base_model = prepare_model_for_kbit_training(base_model)
else:
    model_dtype = torch.float16 if device.type == "cuda" else torch.float32
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, dtype=model_dtype
    ).to(device)

base_model.config.use_cache = False
base_parameter_count = sum(parameter.numel() for parameter in base_model.parameters())
print(f"Base parameters: {base_parameter_count / 1e6:.2f}M")
print(f"Reported memory footprint: {base_model.get_memory_footprint() / 1e6:.2f} MB")

## 7. Add LoRA Adapters

We target only query and value projections. Larger target sets—including key/output projections and feed-forward layers—provide more adaptation capacity at a higher parameter and memory cost.

In [ ]:
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_fraction = trainable_parameters / total_parameters

trainable_names = [
    name for name, parameter in model.named_parameters() if parameter.requires_grad
]
assert trainable_parameters > 0
assert trainable_fraction < 0.01
assert all("lora_" in name for name in trainable_names)
print(f"Exact trainable fraction: {100 * trainable_fraction:.4f}%")
print("First trainable tensors:", trainable_names[:6])

## 8. Generation Helper and Pre-Training Sample

In [ ]:
@torch.no_grad()
def generate_summary(current_model, max_new_tokens=None):
    current_model.eval()
    model_device = current_model.get_input_embeddings().weight.device
    inputs = tokenizer(fixed_prompt, return_tensors="pt").to(model_device)
    output = current_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens or (16 if fast_dev_run else 64),
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

print(generate_summary(model))

## 9. Fine-Tune with SFTTrainer

The adapter learning rate is higher than a typical full-model fine-tuning rate because only newly introduced parameters are optimized. Packing is disabled so prompt/completion boundaries remain easy to inspect.

In [ ]:
training_arguments = SFTConfig(
    output_dir=str(output_dir / "trainer"),
    max_steps=max_steps,
    per_device_train_batch_size=per_device_batch_size,
    per_device_eval_batch_size=per_device_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    warmup_steps=0 if fast_dev_run else 5,
    logging_steps=1 if fast_dev_run else 10,
    eval_strategy="steps",
    eval_steps=1 if fast_dev_run else 25,
    save_strategy="no",
    report_to="none",
    fp16=device.type == "cuda",
    use_cpu=device.type != "cuda",
    max_length=max_length,
    completion_only_loss=True,
    packing=False,
    seed=SEED,
    data_seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_arguments,
    train_dataset=train_dataset,
    eval_dataset=evaluation_dataset,
    processing_class=tokenizer,
)

before_metrics = trainer.evaluate()
train_result = trainer.train()
after_metrics = trainer.evaluate()

print("Evaluation loss before:", before_metrics["eval_loss"])
print("Evaluation loss after: ", after_metrics["eval_loss"])
assert math.isfinite(before_metrics["eval_loss"])
assert math.isfinite(after_metrics["eval_loss"])
assert all(
    parameter.grad is None
    for name, parameter in model.named_parameters()
    if "lora_" not in name
)

if torch.cuda.is_available():
    print(f"Peak allocated GPU memory: {torch.cuda.max_memory_allocated() / 1e6:.2f} MB")

### Training Metrics

Evaluation loss summarizes the held-out completion objective. The training curve shows optimization progress but is not evidence of general-purpose summarization quality.

In [ ]:
comparison_table = pd.DataFrame([
    {"stage": "Before adaptation", "evaluation_loss": before_metrics["eval_loss"]},
    {"stage": "After adaptation", "evaluation_loss": after_metrics["eval_loss"]},
]).set_index("stage")
display(comparison_table.round(4))

history_table = pd.DataFrame(trainer.state.log_history)
if "loss" in history_table.columns:
    training_rows = history_table.dropna(subset=["loss"])
    if not training_rows.empty:
        plt.figure(figsize=(7, 4))
        plt.plot(training_rows["step"], training_rows["loss"], marker="o")
        plt.xlabel("Optimization step")
        plt.ylabel("Completion loss")
        plt.title("LoRA training curve")
        plt.grid(alpha=0.3)
        plt.show()

## 10. Save the Adapter

`save_pretrained` on a PEFT model writes the adapter configuration and adapter weights—not a second copy of the frozen base model.

In [ ]:
model.save_pretrained(adapter_dir, safe_serialization=True)
tokenizer.save_pretrained(adapter_dir)

def directory_size_mb(path):
    path = Path(path)
    return sum(file.stat().st_size for file in path.rglob("*") if file.is_file()) / 1e6

adapter_size_mb = directory_size_mb(adapter_dir)
base_fp16_estimate_mb = base_parameter_count * 2 / 1e6
print(f"Adapter directory: {adapter_size_mb:.2f} MB")
print(f"Estimated FP16 base weights: {base_fp16_estimate_mb:.2f} MB")
assert adapter_size_mb < 0.05 * base_fp16_estimate_mb

print("AFTER ADAPTATION")
print(generate_summary(model))

## 11. Reload and Merge

For a clean comparison, we release the training model, reload the base in float16, attach the saved adapter, and merge `B @ A` into the base weights. Active-adapter and merged logits should agree up to float16 rounding. The merged artifact is full-sized.

In [ ]:
del trainer, model, base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

reload_dtype = torch.float16 if device.type == "cuda" else torch.float32
full_precision_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, dtype=reload_dtype
).to(device)
active_adapter_model = PeftModel.from_pretrained(
    full_precision_base, adapter_dir
).to(device)
active_adapter_model.eval()

check_inputs = tokenizer(fixed_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    active_logits = active_adapter_model(**check_inputs).logits.float().cpu()

merged_model = active_adapter_model.merge_and_unload()
merged_model.eval()
with torch.no_grad():
    merged_logits = merged_model(**check_inputs).logits.float().cpu()

maximum_logit_difference = (active_logits - merged_logits).abs().max().item()
print(f"Maximum active/merged logit difference: {maximum_logit_difference:.6f}")
assert torch.allclose(active_logits, merged_logits, rtol=1e-2, atol=1e-2)

merged_model.save_pretrained(merged_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_dir)
merged_size_mb = directory_size_mb(merged_dir)
print(f"Adapter artifact: {adapter_size_mb:.2f} MB")
print(f"Merged artifact:  {merged_size_mb:.2f} MB")
assert merged_size_mb > 10 * adapter_size_mb

print("MERGED MODEL")
print(generate_summary(merged_model))

## 12. What Was Reduced?

LoRA reduced trainable parameters, gradient storage, optimizer state, and the per-task artifact. It did **not** reduce the parameter count required for ordinary inference. QLoRA additionally reduced the frozen base's training-time memory by storing it in 4-bit form. After merging into a float16 base, that quantization saving is gone unless the merged model is quantized again.

## 13. Things to Try

- Set `USE_QLORA=0` and compare peak GPU memory with the default QLoRA run.
- Sweep ranks `4`, `8`, `16`, and `32`; plot evaluation loss against adapter size.
- Add `k_proj`, `o_proj`, or feed-forward projections to `TARGET_MODULES`.
- Train separate summarization and rewriting adapters, then swap them on the same base.
- Compare merging for deployment with keeping adapters separate for multi-tenant serving.

## Troubleshooting

- If bitsandbytes reports no compatible backend, enable a CUDA runtime or set `USE_QLORA=0`.
- If memory is exhausted, lower the per-device batch size before shortening sequences.
- If the dataset schema changes, inspect one streamed row and preserve the final assistant response as the completion.
- If completion-only loss fails, verify that the records retain conversational `prompt` and `completion` lists and that the instruct tokenizer has a chat template.
- Small before/after generation differences are expected after only two smoke-test steps.

**References:** [LoRA](https://arxiv.org/abs/2106.09685), [QLoRA](https://arxiv.org/abs/2305.14314), [PEFT LoRA](https://huggingface.co/docs/peft/package_reference/lora), [TRL SFTTrainer](https://huggingface.co/docs/trl/sft_trainer), and [SmolTalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk).